# Export SemEval local-evidence scores

This notebook reuses a saved local-evidence classifier to add SemEval window scores to the legacy `scored_windows.csv`. It does not train a model or run language-model inference.

In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import replace
from pathlib import Path

import pandas as pd

from notebooks.utils.feature_aggregation_utils import FeatureSet, ensure_prepared_partition, load_prepared_partition
from notebooks.utils.local_evidence_artifact_utils import load_local_evidence_artifact
from notebooks.utils.local_evidence_evaluation_utils import evaluate_local_evidence_semeval
from notebooks.utils.semeval_local_evidence_export_utils import (
    SCORED_WINDOW_KEY_COLUMNS,
    merge_scored_window_exports,
    validate_scored_window_columns,
)

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
SCORER_MODEL = os.environ.get("LOCAL_EVIDENCE_SCORER_MODEL", "gemma_3_270m_it")
ARTIFACT_RUN_ID = os.environ.get(
    "LOCAL_EVIDENCE_ARTIFACT_RUN_ID",
    f"mixed-bbc-all-llms-2026-09-10-v2-{SCORER_MODEL}",
)
CLASSIFIER_NAME = os.environ.get("LOCAL_EVIDENCE_CLASSIFIER", "logistic_regression")
WINDOW_LENGTHS = (32, 64)
SEMEVAL_DATASET = os.environ.get("LOCAL_EVIDENCE_SEMEVAL_DATASET", "semeval2024_task8_subtask_c_train")
SEMEVAL_SPLIT = os.environ.get("LOCAL_EVIDENCE_SEMEVAL_SPLIT", "MG")
STRIDE = 16
ARTIFACT_PATH = DATA_ROOT / "experiment-results/local-evidence-classification" / ARTIFACT_RUN_ID
SEMEVAL_ANNOTATIONS_PATH = DATA_ROOT / "raw/semevals_2024-task8/SemEval2024-task8/subtaskC/SemEval2024_task8_C_train_MG.csv"
OUTPUT_PATH = Path(os.environ.get(
    "LOCAL_EVIDENCE_SCORED_WINDOWS_PATH",
    str(ARTIFACT_PATH / "csv_exports/scored_windows.csv"),
))
print({"artifact": str(ARTIFACT_PATH), "annotations": str(SEMEVAL_ANNOTATIONS_PATH), "output": str(OUTPUT_PATH)})

In [ ]:
if not SEMEVAL_ANNOTATIONS_PATH.exists():
    raise FileNotFoundError(f"SemEval annotations not found: {SEMEVAL_ANNOTATIONS_PATH}")
metadata = json.loads((ARTIFACT_PATH / "metadata.json").read_text(encoding="utf-8"))
artifact = load_local_evidence_artifact(
    DATA_ROOT,
    ARTIFACT_RUN_ID,
    expected_metadata=metadata["metadata"],
    trust_artifact=True,
)
available_keys = set(artifact.model.classifiers)
requested_keys = {(CLASSIFIER_NAME, int(length)) for length in WINDOW_LENGTHS}
missing_keys = sorted(requested_keys.difference(available_keys))
if missing_keys:
    raise ValueError(f"artifact is missing classifier/window configurations: {missing_keys}")
selected_classifiers = {key: artifact.model.classifiers[key] for key in requested_keys}
selected_model = replace(
    artifact.model,
    classifiers=selected_classifiers,
    window_lengths=tuple(WINDOW_LENGTHS),
)
annotations = pd.read_csv(SEMEVAL_ANNOTATIONS_PATH)
if "id" not in annotations.columns:
    raise ValueError("SemEval annotations must contain an id column")
print(f"Loaded {len(annotations):,} SemEval annotations and {len(selected_classifiers)} classifier/window models.")

In [ ]:
prepared_frames = []
for window_length in WINDOW_LENGTHS:
    feature_set = FeatureSet("artifact_features", artifact.model.feature_columns)
    prepared = ensure_prepared_partition(
        DATA_ROOT,
        DATA_ROOT / "processed",
        scorer_llm=SCORER_MODEL,
        dataset=SEMEVAL_DATASET,
        split=SEMEVAL_SPLIT,
        feature_set=feature_set,
        window_configuration=window_length,
        stride=STRIDE,
    )
    if prepared["status"] != "available":
        raise ValueError(f"SemEval prepared partition unavailable: {prepared.get('error', prepared)}")
    expected = {
        "schema_version": 1,
        "scorer_llm": SCORER_MODEL,
        "dataset": SEMEVAL_DATASET,
        "split": SEMEVAL_SPLIT,
        "feature_set": "artifact_features",
        "feature_columns": list(artifact.model.feature_columns),
        "window_configuration": str(window_length),
        "stride": STRIDE,
    }
    frame = load_prepared_partition(prepared["path"], expected=expected).copy()
    frame["window_length"] = int(window_length)
    prepared_frames.append(frame)
prepared_windows = pd.concat(prepared_frames, ignore_index=True)
result = evaluate_local_evidence_semeval(
    selected_model,
    DATA_ROOT / "processed",
    annotations,
    dataset=SEMEVAL_DATASET,
    split=SEMEVAL_SPLIT,
    windows_override=prepared_windows,
)
generated = result.scored_windows.copy()
if generated.empty:
    raise ValueError("SemEval evaluation produced no scored windows")
generated["dataset"] = SEMEVAL_DATASET
generated["split"] = SEMEVAL_SPLIT
generated["llm"] = SCORER_MODEL
generated["training_regime"] = metadata["metadata"].get("training_regime", ARTIFACT_RUN_ID)
generated["partition"] = "test"
generated["evaluation_dataset"] = SEMEVAL_DATASET
generated["evaluation_regime"] = "SemEval"
print(generated.groupby(["window_length", "model"]).size().to_string())

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
existing = pd.read_csv(OUTPUT_PATH) if OUTPUT_PATH.exists() else pd.DataFrame(columns=generated.columns)
if not existing.empty:
    generated = generated.reindex(columns=existing.columns)
else:
    existing = existing.reindex(columns=generated.columns)
validate_scored_window_columns(existing, SCORED_WINDOW_KEY_COLUMNS)
validate_scored_window_columns(generated, SCORED_WINDOW_KEY_COLUMNS)
merged = merge_scored_window_exports(existing, generated)
tmp_path = OUTPUT_PATH.with_suffix(".tmp.csv")
merged.to_csv(tmp_path, index=False)
tmp_path.replace(OUTPUT_PATH)
semeval_rows = merged.loc[merged["dataset"].eq(SEMEVAL_DATASET)]
print(f"Existing rows: {len(existing):,}")
print(f"Generated SemEval rows: {len(generated):,}")
print(f"Final rows: {len(merged):,}")
print(semeval_rows.groupby(["window_length", "model"]).size().to_string())